# Deficit de Vivienda - Barrio Obrero (Cali)

Mapa de **Deficit de Vivienda** del ITT. 4 capas base (OSM activa), capas apagadas salvo el
limite, panel colapsable. Salida: `Barrio_obrero_actual/mapa_itt_barrio_obrero_06_deficit_vivienda.html`.

## 0. Setup Colab (opcional)

In [ ]:
def _en_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False
if _en_colab():
    import os,subprocess,sys
    REPO='https://github.com/j0rg3c45/mapa_Analitico_ciudad_Paraiso.git'
    DEST='/content/mapa_Analitico_ciudad_Paraiso'
    subprocess.run([sys.executable,'-m','pip','install','-q','geopandas','folium','mapclassify','pyarrow','openpyxl'],check=False)
    if not os.path.isdir(DEST): subprocess.run(['git','clone','--depth','1',REPO,DEST],check=False)
    if os.path.isdir(os.path.join(DEST,'notebooks')): os.chdir(os.path.join(DEST,'notebooks'))
    print('Colab listo')
else:
    print('Entorno local (uv).')


## 1. Setup, datos y helpers

In [ ]:
import os, json, math
import numpy as np, pandas as pd, geopandas as gpd
import folium
from folium.plugins import HeatMap
# Paleta Okabe-Ito (cualitativas) + cividis (cuantitativas/calor).
OKI_AZUL="#0072B2"; OKI_NARANJA="#E69F00"; OKI_VERDE="#009E73"; OKI_AMARILLO="#F0E442"
OKI_AZUL_C="#56B4E9"; OKI_BERMELL="#D55E00"; OKI_MORADO="#CC79A7"; OKI_GRIS="#333333"
BG="#F4F6F9"; CRS_WEB=4326; LIMITE="#1B4F8A"
# Gradiente CIVIDIS (azul oscuro -> amarillo), apto daltonismo, para TODOS los heatmaps.
GRAD_CIVIDIS={0.0:"#00204d",0.25:"#31446b",0.5:"#666970",0.75:"#a69d75",1.0:"#fdea45"}
def _ruta(*cands):
    for p in cands:
        if os.path.isfile(p) or os.path.isdir(p): return os.path.abspath(p)
    return None
def _en(d,pref):
    if not d: return None
    for f in os.listdir(d):
        if f.startswith(pref) and f.endswith(".geojson"): return os.path.join(d,f)
    return None
DATADIR=_ruta(os.path.join("..","data","Barrio_Obrero_geojson","barrio_obrero_completo"),
              os.path.join("data","Barrio_Obrero_geojson","barrio_obrero_completo"),
              os.path.join("/content","data","Barrio_Obrero_geojson","barrio_obrero_completo"))
POLY=_ruta(os.path.join("..","data","Barrio_Obrero_geojson","Geojson_Barrio_Obrero.geojson"),
           os.path.join("data","Barrio_Obrero_geojson","Geojson_Barrio_Obrero.geojson"),
           os.path.join("/content","data","Barrio_Obrero_geojson","Geojson_Barrio_Obrero.geojson"))
MAESTRO=_ruta(os.path.join("..","data","Barrio_Obrero_geojson","itt_maestro_barrio_obrero.json"),
              os.path.join("data","Barrio_Obrero_geojson","itt_maestro_barrio_obrero.json"),
              os.path.join("/content","data","Barrio_Obrero_geojson","itt_maestro_barrio_obrero.json"))
assert DATADIR and POLY and MAESTRO, "faltan datos de Barrio Obrero o el JSON maestro"
MASTER=json.load(open(MAESTRO,encoding="utf-8"))
POLIGONO_ZONA=gpd.read_file(POLY).to_crs(CRS_WEB)
_b=POLIGONO_ZONA.total_bounds; BO_BOUNDS=[[float(_b[1]),float(_b[0])],[float(_b[3]),float(_b[2])]]
OUT_DIR=None
for _d in [os.path.join("..","data_processed_geo"),"data_processed_geo","/content/data_processed_geo"]:
    try: os.makedirs(_d,exist_ok=True); OUT_DIR=os.path.abspath(_d); break
    except Exception: pass
MAPAS_DIR=os.path.abspath(os.path.join(os.path.dirname(OUT_DIR.rstrip(os.sep)),"mapa_html","Barrio_obrero","Barrio_obrero_actual"))
os.makedirs(MAPAS_DIR,exist_ok=True)
def cargar_pts(pref, filtro_col=None, filtro_val=None):
    p=_en(DATADIR,pref)
    if p is None: return gpd.GeoDataFrame(geometry=[],crs=CRS_WEB)
    g=gpd.read_file(p).to_crs(CRS_WEB)
    g=g[g.geometry.notna() & g.geom_type.str.contains("Point")].copy()
    if filtro_col and filtro_col in g.columns:
        g=g[g[filtro_col].astype(str).str.strip().str.upper()==filtro_val].copy()
    if len(g):
        g["lat"]=g.geometry.y; g["lon"]=g.geometry.x
    else:
        g["lat"]=[]; g["lon"]=[]
    return g
def base_map(zoom=15):
    _c=POLIGONO_ZONA.geometry.union_all().centroid
    m=folium.Map(location=[_c.y,_c.x],zoom_start=zoom,tiles=None,control_scale=True,
                 zoom_snap=0.25,zoom_delta=0.1,max_zoom=22)
    m.get_root().width="100%"; m.get_root().height="100%"
    m.get_root().header.add_child(folium.Element(
        "<style>html,body{height:100%;margin:0;padding:0}"
        ".folium-map{position:absolute;top:0;bottom:0;left:0;right:0;width:100%;height:100%}"
        ".leaflet-top,.leaflet-bottom{z-index:1000}"
        "#dim-panel{z-index:1001;background:rgba(255,255,255,.95);backdrop-filter:blur(4px);"
        "-webkit-backdrop-filter:blur(4px);box-shadow:0 4px 15px rgba(0,0,0,.2)}</style>"))
    # 4 capas base: OSM (activa), Esri Satelite, Google Streets, Google Satelite.
    folium.TileLayer("OpenStreetMap",name="OpenStreetMap",show=True,max_zoom=22).add_to(m)
    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",attr="Esri",name="Esri Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
    folium.TileLayer("https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",attr="Google",name="Google Streets",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
    folium.TileLayer("https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",attr="Google",name="Google Satelite",show=False,max_zoom=22,max_native_zoom=20).add_to(m)
    _zg=POLIGONO_ZONA.geometry.union_all(); _mundo=[[-180,-85],[180,-85],[180,85],[-180,85],[-180,-85]]; _ag=[]
    for _gm in (list(_zg.geoms) if _zg.geom_type=="MultiPolygon" else [_zg]):
        _ag.append([[x,y] for x,y in _gm.exterior.coords])
    folium.GeoJson({"type":"Feature","properties":{},"geometry":{"type":"Polygon","coordinates":[_mundo]+_ag}},
        name="Mascara (foco en la zona)",control=False,
        style_function=lambda f:{"fillColor":"#000","color":"#000","weight":0,"fillOpacity":0.18}).add_to(m)
    # Limite perimetral: UNICA capa activa al inicio.
    folium.GeoJson(POLIGONO_ZONA.to_json(),name="Limite Barrio Obrero",show=True,
        style_function=lambda f:{"color":LIMITE,"weight":2.5,"fill":False}).add_to(m)
    return m
def add_puntos(m, g, nombre, color, borde=None, radio=3.0, show=False, popup_cols=None):
    fg=folium.FeatureGroup(name="Puntos: "+nombre+" ("+str(len(g))+")",show=show)
    bc=borde or color
    for _,r in g.iterrows():
        pop=None
        if popup_cols:
            filas="".join("<div><b>%s:</b> %s</div>"%(lbl,r.get(col,"s/d")) for lbl,col in popup_cols if col in g.columns)
            pop=folium.Popup('<div style="font-family:system-ui;font-size:12px;background:'+BG+';padding:6px 8px;border-radius:6px;min-width:150px">'+filas+'</div>',max_width=220)
        folium.CircleMarker([r["lat"],r["lon"]],radius=radio,color=bc,weight=0.6,
            fill=True,fill_color=color,fill_opacity=0.78,popup=pop).add_to(fg)
    fg.add_to(m); return fg
def add_calor(m, g, nombre, show=False):
    # TODOS los heatmaps usan cividis (cuantitativo/termico, apto daltonismo).
    pts=g[["lat","lon"]].dropna().values.tolist() if len(g) else []
    fg=folium.FeatureGroup(name="Calor: "+nombre+" ("+str(len(pts))+")",show=show)
    if pts:
        HeatMap(pts,gradient=GRAD_CIVIDIS,radius=18,blur=14,min_opacity=0.35,name="heat").add_to(fg)
    fg.add_to(m); return fg
def _slider(m):
    mid=m.get_name()
    html=("<div id='heat-intensity-ctrl' style='position:fixed;bottom:24px;right:24px;z-index:1000;"
          "background:"+BG+";padding:8px 12px;border-radius:10px;box-shadow:0 2px 10px rgba(0,0,0,.3);"
          "font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:"+OKI_GRIS+";max-width:210px'>"
          "<div style='font-weight:700;margin-bottom:4px'>Intensidad del calor</div>"
          "<input id='heat-range' type='range' min='8' max='40' value='18' style='width:100%;cursor:pointer'>"
          "<div style='font-size:10px;color:#666;margin-top:2px'>Radio: <span id='heat-val'>18</span> px</div></div>"
          "<script>(function(){function ap(v){var mp=window['"+mid+"'];if(!mp||!mp.eachLayer)return;"
          "mp.eachLayer(function(l){if(l&&l.setOptions&&l._heat){try{l.setOptions({radius:+v,blur:Math.round(v*0.8)});l.redraw&&l.redraw();}catch(e){}}});}"
          "function init(){var r=document.getElementById('heat-range'),o=document.getElementById('heat-val');"
          "if(!r)return;r.addEventListener('input',function(){o.textContent=r.value;ap(r.value);});}"
          "if(document.readyState==='complete'){setTimeout(init,400);}else{window.addEventListener('load',function(){setTimeout(init,400);});}})();</script>")
    m.get_root().html.add_child(folium.Element(html))
print("Setup OK (4 bases + OSM, cividis). Salida:",MAPAS_DIR)


## 2. Construccion del mapa

In [ ]:
m = base_map(zoom=15)
# Coropleta SISBEN por poligono (cividis segun % deficit), recortada a la zona.
DEF=_ruta(os.path.join("..","data","5_Dimension_Entorno_Urbano","sec_vivienda","DEF_HAB_SISBEN.shp"),
          os.path.join("data","5_Dimension_Entorno_Urbano","sec_vivienda","DEF_HAB_SISBEN.shp"),
          os.path.join("/content","data","5_Dimension_Entorno_Urbano","sec_vivienda","DEF_HAB_SISBEN.shp"))
# Cividis discreta para % deficit.
def _civ(p):
    if p<30: return "#00204d"
    if p<38: return "#414d6b"
    if p<44: return "#7c7b78"
    if p<50: return "#bcaf6f"
    return "#fdea45"
if DEF:
    d=gpd.read_file(DEF).to_crs(CRS_WEB)
    _zu=POLIGONO_ZONA.geometry.union_all()
    d=d[d.geometry.intersects(_zu)].copy()
    d["F_HOG_DE_1"]=pd.to_numeric(d["F_HOG_DE_1"],errors="coerce")
    d["T_HOG"]=pd.to_numeric(d["T_HOG"],errors="coerce").fillna(0)
    fgd=folium.FeatureGroup(name="Deficit habitacional por sector (SISBEN)",show=False)
    for _,r in d.iterrows():
        th=r["T_HOG"]; pct=r["F_HOG_DE_1"]
        fill="#D6D9DF" if th<5 or pd.isna(pct) else _civ(float(pct))
        pop=('<div style="font-family:system-ui;font-size:12px;background:'+BG+';padding:7px 9px;border-radius:6px;min-width:190px">'
             '<div style="font-weight:700;color:#54278f">Deficit habitacional (SISBEN)</div>'
             '<div>Hogares evaluados: <b>%d</b></div><div>Tasa de deficit: <b>%s</b></div></div>'%(int(th),("%.1f%%"%pct) if pd.notna(pct) else "s/d"))
        folium.GeoJson(r["geometry"].__geo_interface__,
            style_function=(lambda f,c=fill:{"fillColor":c,"color":"#fff","weight":0.8,"fillOpacity":0.8}),
            tooltip=("deficit %.0f%%"%pct) if pd.notna(pct) and th>=5 else "muestra pequena",
            popup=folium.Popup(pop,max_width=240)).add_to(fgd)
    fgd.add_to(m)
# Datos oficiales por barrio (Excel BD_DEFICIT_HABITACIONAL_BARRIOS_2024) para el panel.
XLS=_ruta(os.path.join("..","data","5_Dimension_Entorno_Urbano","sec_vivienda","BD_DEFICIT_HABITACIONAL_BARRIOS_2024.xlsx"),
          os.path.join("data","5_Dimension_Entorno_Urbano","sec_vivienda","BD_DEFICIT_HABITACIONAL_BARRIOS_2024.xlsx"),
          os.path.join("/content","data","5_Dimension_Entorno_Urbano","sec_vivienda","BD_DEFICIT_HABITACIONAL_BARRIOS_2024.xlsx"))
BO={}
if XLS:
    _df=pd.read_excel(XLS)
    _m=_df[_df["NOMBRE_LOC"].astype(str).str.upper().str.contains("OBRERO",na=False)]
    if len(_m):
        BO=_m.iloc[0].to_dict()
_tot=int(BO.get("Total de hogares por barrio",0) or 0)
_def=int(BO.get("Déficit Habitacional",0) or 0)
_cuant=int(BO.get("Déficit Cuantitativo",0) or 0)
_cual=int(BO.get("Déficit Cualitativo",0) or 0)
_tasa=(100.0*_def/_tot) if _tot else 0.0
# Componentes para el grafico de barras.
_comp=[("Cocina",BO.get("Déficit cualitativo Cocina")),("Hacinamiento mit.",BO.get("Déficit cualitativo Hacinamiento mitigable")),
       ("Recol. basuras",BO.get("Déficit cualitativo Recolección de basuras")),("Acueducto",BO.get("Déficit cualitativo Acueducto")),
       ("Paredes",BO.get("Déficit cuantitativo Tipo de paredes")),("Hacin. no mit.",BO.get("Déficit cuantitativo Hacinamiento no mitigable")),
       ("Alcantarillado",BO.get("Déficit cualitativo Alcantarillado")),("Cohabitacion",BO.get("Déficit cuantitativo Cohabitación"))]
_comp=[(k,int(v or 0)) for k,v in _comp]
_chart=json.dumps({"labels":[k for k,_ in _comp],"data":[v for _,v in _comp]},ensure_ascii=False)
_panel=r"""
<script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
<div id="dim-panel" style="position:fixed;bottom:25px;left:20px;z-index:1001;max-width:400px;width:min(400px,44vw);max-height:84vh;overflow-y:auto;padding:11px 13px;border-radius:10px;font-family:system-ui,Segoe UI,sans-serif;font-size:12px;color:__GRIS__">
  <div style="display:flex;align-items:center;justify-content:space-between;gap:8px">
    <div style="font-weight:700;font-size:13px">Deficit de vivienda - Barrio Obrero</div>
    <button id="dim-toggle" style="cursor:pointer;border:1px solid #bbb;background:#fff;color:#333;width:24px;height:22px;border-radius:5px;font-size:13px;font-weight:700;flex:0 0 auto">_</button>
  </div>
  <div id="dim-body">
    <div style="font-size:10px;color:#666;margin:4px 0 6px">Deficit habitacional 2024 (SISBEN, por barrio). Fuente: Alcaldia de Cali.</div>
    <div style="font-size:11px;margin:2px 0"><b>__TOT__</b> hogares &nbsp;|&nbsp; deficit <b>__DEF__</b> (<b>__TASA__%</b>)</div>
    <div style="font-size:11px;margin:1px 0">Cuantitativo: <b>__CUANT__</b> &nbsp;|&nbsp; Cualitativo: <b>__CUAL__</b></div>
    <details open style="margin-top:8px">
      <summary style="cursor:pointer;font-weight:600;color:__AZUL__;outline:none">Componentes del deficit / Ocultar</summary>
      <div style="position:relative;height:220px;width:100%;margin-top:6px"><canvas id="dim-chart"></canvas></div>
      <div style="font-size:9px;color:#666;margin-top:3px">Eje: Cantidad de hogares afectados por componente.</div>
    </details>
  </div>
</div>
<script>
(function(){
  var D=__CHART__;
  function dib(){var cv=document.getElementById("dim-chart");if(!cv||typeof Chart==="undefined")return;
    new Chart(cv.getContext("2d"),{type:"bar",data:{labels:D.labels,datasets:[{data:D.data,backgroundColor:"#54278f"}]},
      options:{indexAxis:"y",responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{beginAtZero:true,title:{display:true,text:"Cantidad",font:{size:9}},ticks:{font:{size:9}}},y:{ticks:{font:{size:8}}}}}});}
  function initC(){var b=document.getElementById("dim-toggle"),bo=document.getElementById("dim-body");if(!b||!bo)return;b.addEventListener("click",function(){var o=(bo.style.display==="none");bo.style.display=o?"":"none";b.textContent=o?"_":"+";});}
  if(document.readyState==="complete"){setTimeout(dib,300);initC();}else{window.addEventListener("load",function(){setTimeout(dib,300);initC();});}
})();
</script>
"""
_panel=(_panel.replace("__GRIS__",OKI_GRIS).replace("__AZUL__",OKI_AZUL).replace("__CHART__",_chart)
             .replace("__TOT__",str(_tot)).replace("__DEF__",str(_def))
             .replace("__TASA__","%.1f"%_tasa).replace("__CUANT__",str(_cuant)).replace("__CUAL__",str(_cual)))
m.get_root().html.add_child(folium.Element(_panel))
# Leyenda cividis.
_ley='<div style="position:fixed;top:12px;right:12px;z-index:1000;background:'+BG+';padding:8px 10px;border-radius:8px;box-shadow:0 2px 8px rgba(0,0,0,.25);font-family:system-ui;color:'+OKI_GRIS+';max-width:200px;font-size:10px"><div style="font-weight:700;font-size:11px;margin-bottom:4px">% hogares en deficit</div>'
for _l,_c in [("< 30%","#00204d"),("30-38%","#414d6b"),("38-44%","#7c7b78"),("44-50%","#bcaf6f"),("> 50%","#fdea45"),("muestra pequena","#D6D9DF")]:
    _ley+='<div style="display:flex;align-items:center;gap:6px;margin:1px 0"><i style="width:16px;height:12px;background:%s;display:inline-block;border:1px solid #999"></i>%s</div>'%(_c,_l)
_ley+='</div>'
m.get_root().html.add_child(folium.Element(_ley))
folium.LayerControl(collapsed=True,position="topright").add_to(m)
print("OK deficit vivienda. Barrio Obrero deficit:",_def,"de",_tot,"hogares")

## 3. Exportacion del HTML

In [ ]:
html=os.path.join(MAPAS_DIR,"mapa_itt_barrio_obrero_06_deficit_vivienda.html")
m.save(html)
print("exportado:", html)
m